# R Master v3 · 一键云跑（不用再上传）

直接点 **运行全部 / Run all**，然后允许 Google Drive。

这版直接读取已经生成好的：

`MyDrive/R_Master/v2/latest/R_Master_Align_v2_PREVIEW.blend`

所以：
- 不再上传 Mona；
- 不重新做 v2 的比例计算；
- 不重新下载 Blender（Drive 有缓存就直接复用）；
- 只做 body-shell 诊断渲染，速度会比前面更快；
- 重 `.blend` 留在 Drive，手机只下载一个小的 `R_Master_v3_Review.zip`。



In [ ]:
from google.colab import drive, files
from IPython.display import display, Image, Markdown
from pathlib import Path
import shutil, subprocess, zipfile, json, textwrap, os

BLENDER_VERSION = "4.4.3"
BLENDER_URL = "https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"

LOCAL = Path("/content/r_master_v3")
OUT = LOCAL / "output"
LOCAL_ARCHIVE = LOCAL / f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
LOCAL_BLENDER_DIR = LOCAL / f"blender-{BLENDER_VERSION}-linux-x64"
LOCAL_BLEND = LOCAL / "R_Master_Align_v2_PREVIEW.blend"
LOCAL_SCRIPT = LOCAL / "R_Master_v3_Render.py"
REVIEW = LOCAL / "R_Master_v3_Review.zip"
LOCAL.mkdir(parents=True, exist_ok=True)

print("R Master v3 · Body Shell 一键诊断")
print("① 挂载 Google Drive…")
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/R_Master")
V2 = ROOT / "v2" / "latest" / "R_Master_Align_v2_PREVIEW.blend"
CACHE = ROOT / "cache"
V3 = ROOT / "v3" / "latest"
CACHE.mkdir(parents=True, exist_ok=True)
V3.mkdir(parents=True, exist_ok=True)
DRIVE_ARCHIVE = CACHE / LOCAL_ARCHIVE.name

if not V2.exists() or V2.stat().st_size < 50 * 1024 * 1024:
    raise RuntimeError("没有找到 v2 预览文件：MyDrive/R_Master/v2/latest/R_Master_Align_v2_PREVIEW.blend。请先确认 v2 成功跑完。")

print(f"✓ 读取 v2 预览：{V2.stat().st_size/1024/1024:.1f} MiB")
shutil.copy2(V2, LOCAL_BLEND)

print("② 准备 Blender…")
if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size > 100 * 1024 * 1024:
    print("✓ 复用 Drive 里的 Blender 缓存")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
else:
    print("补下载 Blender 一次…")
    subprocess.run(["wget","-q","--show-progress","-O",str(LOCAL_ARCHIVE),BLENDER_URL], check=True)
    shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

if LOCAL_BLENDER_DIR.exists():
    shutil.rmtree(LOCAL_BLENDER_DIR)
subprocess.run(["tar","-xf",str(LOCAL_ARCHIVE),"-C",str(LOCAL)], check=True)
BLENDER = LOCAL_BLENDER_DIR / "blender"
if not BLENDER.exists():
    raise RuntimeError("Blender 解压失败。")

script = r"""
import bpy, json, os, sys
from mathutils import Vector

def args():
    xs = sys.argv[sys.argv.index("--")+1:] if "--" in sys.argv else []
    out = None
    for i,x in enumerate(xs):
        if x == "--out" and i+1 < len(xs): out = xs[i+1]
    return os.path.abspath(out or "/tmp/r_master_v3")

OUT = args()
os.makedirs(OUT, exist_ok=True)

body = bpy.data.objects.get("R2_Mona_Main") or bpy.data.objects.get("Mona_Main")
if not body or body.type != "MESH":
    raise RuntimeError("找不到 R2_Mona_Main / Mona_Main")

# 只显示 body shell；其他衣物/鞋/配件/控制器不参与渲染
for obj in bpy.context.scene.objects:
    if obj.type == "MESH":
        obj.hide_render = obj != body
        obj.hide_viewport = obj != body
    elif obj.type == "ARMATURE":
        obj.hide_render = True

# 再次确认这些干扰 modifier 不参与预览
disabled = []
for mod in body.modifiers:
    if mod.type in {"MASK","SURFACE_DEFORM","CLOTH"} or "mask" in mod.name.lower() or "cloth" in mod.name.lower():
        if mod.show_viewport or mod.show_render:
            disabled.append({"name":mod.name,"type":mod.type})
        mod.show_viewport = False
        mod.show_render = False
    elif mod.type in {"MULTIRES","SUBSURF"}:
        mod.levels = min(mod.levels, 1)
        mod.render_levels = min(mod.render_levels, 1)

def bounds(obj):
    pts = [obj.matrix_world @ Vector(c) for c in obj.bound_box]
    mn = Vector((min(p.x for p in pts), min(p.y for p in pts), min(p.z for p in pts)))
    mx = Vector((max(p.x for p in pts), max(p.y for p in pts), max(p.z for p in pts)))
    return mn,mx

def look_at(obj,target):
    obj.rotation_euler = (Vector(target)-obj.location).to_track_quat("-Z","Y").to_euler()

scene = bpy.context.scene
scene.render.engine = "BLENDER_WORKBENCH"
scene.render.image_settings.file_format = "PNG"
scene.render.film_transparent = False
scene.display.shading.light = "STUDIO"
scene.display.shading.show_shadows = True
scene.display.shading.show_cavity = True
scene.display.shading.cavity_type = "WORLD"
scene.display.shading.color_type = "SINGLE"
scene.display.shading.single_color = (0.58,0.58,0.61)
scene.display.shading.background_type = "VIEWPORT"
scene.display.shading.background_color = (0.045,0.045,0.055)

cam_data = bpy.data.cameras.get("R_Master_v3_Camera_DATA") or bpy.data.cameras.new("R_Master_v3_Camera_DATA")
cam = bpy.data.objects.get("R_Master_v3_Camera")
if not cam:
    cam = bpy.data.objects.new("R_Master_v3_Camera", cam_data)
    bpy.context.scene.collection.objects.link(cam)
scene.camera = cam
cam.data.type = "ORTHO"

mn,mx = bounds(body)
center = (mn+mx)*0.5
height = mx.z-mn.z
width = mx.x-mn.x
depth = mx.y-mn.y
dist = max(height,width,depth)*2.5

def render(name,pos,target,scale,res=(720,960)):
    scene.render.resolution_x, scene.render.resolution_y = res
    scene.render.resolution_percentage = 100
    cam.location = Vector(pos)
    cam.data.ortho_scale = scale
    look_at(cam, Vector(target))
    path = os.path.join(OUT, name+".png")
    scene.render.filepath = path
    bpy.ops.render.render(write_still=True)
    return path

renders = {}
renders["full_front"] = render("R_Master_v3_full_front",(center.x,center.y-dist,center.z),center,height*1.08)
renders["full_side"] = render("R_Master_v3_full_side",(center.x+dist,center.y,center.z),center,height*1.08)
renders["full_three_quarter"] = render("R_Master_v3_full_three_quarter",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08)

regions = {
    "torso_waist": 0.61,
    "waist_pelvis": 0.51,
    "pelvis_upperthigh": 0.42,
}
for key,frac in regions.items():
    z = mn.z + height*frac
    target = (center.x,center.y,z)
    renders[key] = render("R_Master_v3_"+key,(center.x,center.y-dist,z),target,max(.42,height*.25),(900,700))

z = mn.z + height*.46
renders["glute_side"] = render("R_Master_v3_glute_side",(center.x+dist,center.y,z),(center.x,center.y,z),max(.42,height*.25),(900,700))

report = {
    "ok": True,
    "stage": "R_Master_BodyShellDiagnostic_v3",
    "source": bpy.data.filepath,
    "body_mesh": body.name,
    "bone_count": len((bpy.data.objects.get("R_Master_Align_v2_PREVIEW") or bpy.data.objects.get("Mona_Armature")).data.bones),
    "disabled_modifiers_this_pass": disabled,
    "bounds": {"min":list(mn),"max":list(mx),"center":list(center)},
    "renders": renders,
    "rest_pose_baked": False,
    "final_vrm": False,
}
with open(os.path.join(OUT,"R_Master_v3_report.json"),"w",encoding="utf-8") as f:
    json.dump(report,f,ensure_ascii=False,indent=2)

out_blend = os.path.join(OUT,"R_Master_Align_v3_PREVIEW.blend")
bpy.ops.wm.save_as_mainfile(filepath=out_blend, check_existing=False)
print("[R Master v3] BUILD_OK")
print("[R Master v3] body:", body.name)
print("[R Master v3] blend:", out_blend)
"""

LOCAL_SCRIPT.write_text(script, encoding="utf-8")
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

print("③ 云端 Blender 正在渲染 v3 Body Shell…")
log_path = OUT / "R_Master_v3_blender.log"
cmd = ["xvfb-run","-a",str(BLENDER),"--background",str(LOCAL_BLEND),"--python",str(LOCAL_SCRIPT),"--","--out",str(OUT)]
with log_path.open("w",encoding="utf-8") as log:
    proc = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        log.write(line)
        if "R Master v3" in line or "Error" in line or "Traceback" in line:
            print(line.rstrip())
    code = proc.wait()

if code != 0:
    print(log_path.read_text(encoding="utf-8",errors="replace")[-8000:])
    raise RuntimeError(f"v3 运行失败，退出码 {code}。截图给二蛋即可。")

report_path = OUT / "R_Master_v3_report.json"
blend_path = OUT / "R_Master_Align_v3_PREVIEW.blend"
imgs = [
    OUT/"R_Master_v3_full_front.png",
    OUT/"R_Master_v3_full_side.png",
    OUT/"R_Master_v3_full_three_quarter.png",
    OUT/"R_Master_v3_torso_waist.png",
    OUT/"R_Master_v3_waist_pelvis.png",
    OUT/"R_Master_v3_pelvis_upperthigh.png",
    OUT/"R_Master_v3_glute_side.png",
]
missing = [p.name for p in [report_path,blend_path,log_path,*imgs] if not p.exists()]
if missing:
    raise RuntimeError("v3 缺少输出："+", ".join(missing))

print("④ 保存重文件到 Drive…")
for p in V3.iterdir():
    if p.is_file(): p.unlink()
for p in OUT.iterdir():
    if p.is_file(): shutil.copy2(p,V3/p.name)
print("✓ MyDrive/R_Master/v3/latest/")

labels = ["全身正面","全身侧面","全身 3/4","胸廓→腰","腰→骨盆","骨盆→大腿根","臀线侧视"]
for title,path in zip(labels,imgs):
    display(Markdown(f"### {title}"))
    display(Image(filename=str(path),width=500))

print("⑤ 打包轻量 Review ZIP…")
if REVIEW.exists(): REVIEW.unlink()
with zipfile.ZipFile(REVIEW,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in [*imgs,report_path,log_path]:
        z.write(p,arcname=p.name)
shutil.copy2(REVIEW,V3/REVIEW.name)
print(f"✓ R_Master_v3_Review.zip · {REVIEW.stat().st_size/1024/1024:.1f} MiB")
files.download(str(REVIEW))

